# AED RH - Notebook d'analyse exploratoire et predictive

Notebook pedagogique construit pour l'examen blanc AED RH. Toutes les analyses restent au niveau collectif equipe/site/metier.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

ROOT = Path('..')
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

## 1. Chargement des 4 fichiers CSV

Objectif metier : comprendre les sources disponibles avant toute transformation.

In [ ]:
workforce = pd.read_csv(RAW / 'workforce_BIG.csv')
reference = pd.read_csv(RAW / 'reference_BIG.csv')
skills = pd.read_csv(RAW / 'skills_BIG.csv')
recruitment = pd.read_csv(RAW / 'recruitment_BIG.csv')
files = {'workforce': workforce, 'reference': reference, 'skills': skills, 'recruitment': recruitment}
for name, df in files.items():
    print(name, df.shape)
    display(df.head())

## 2. Dimensions, types, valeurs manquantes et doublons

Cette etape sert a reperer les risques de fiabilite avant construction d'indicateurs RH.

In [ ]:
for name, df in files.items():
    print('\n---', name, '---')
    print(df.dtypes)
    print('Valeurs manquantes:', int(df.isna().sum().sum()))
    print('Doublons:', int(df.duplicated().sum()))

## 3. Incoherences metier et valeurs extremes

Les controles portent sur les effectifs negatifs, ratios hors bornes et delais incoherents.

In [ ]:
checks = {
    'workforce_effectif_reel_negatif': (pd.to_numeric(workforce['headcount_actual'], errors='coerce') < 0).sum(),
    'workforce_attrition_hors_borne_absenteisme': (~pd.to_numeric(workforce['absenteeism_rate'], errors='coerce').between(0, 1)).sum(),
    'workforce_couverture_hors_borne': (~pd.to_numeric(workforce['critical_skill_coverage_rate'], errors='coerce').between(0, 1)).sum(),
    'recruitment_acceptation_hors_borne': (~pd.to_numeric(recruitment['offer_acceptance_rate'], errors='coerce').between(0, 1)).sum(),
    'skills_besoins_negatifs': (pd.to_numeric(skills['required_people_count'], errors='coerce') < 0).sum(),
}
pd.Series(checks).sort_values(ascending=False)

## 4. Visualisations AED

Les graphiques donnent une premiere lecture des dynamiques collectives.

In [ ]:
wf_plot = workforce.copy()
wf_plot['month'] = pd.to_datetime(wf_plot['month'], errors='coerce')
wf_plot['site'] = wf_plot['site'].astype(str).str.strip().str.title()
wf_plot.groupby('site')['headcount_actual'].sum().sort_values().plot(kind='barh', title='Effectif reel total par site')
plt.show()
wf_plot.groupby('site')['voluntary_leavers'].sum().sort_values().plot(kind='barh', title='Departs volontaires par site')
plt.show()

## 5. Nettoyage, jointures et DATA GOLD

Pour garantir la reproductibilite, le projet genere la DATA GOLD via `build_project.py`. Les jointures restent agregees.

In [ ]:
gold = pd.read_csv(PROCESSED / 'gold_data_rh.csv')
data_dictionary = pd.read_csv(PROCESSED / 'data_dictionary.csv')
quality = pd.read_csv(PROCESSED / 'dashboard_data_quality.csv')
display(gold.head())
display(data_dictionary.head(12))
display(quality)

## 6. Creation des KPI RH

Les KPI principaux sont l'ecart d'effectif, le taux d'attrition, la tension recrutement, la couverture skills et le risque collectif.

In [ ]:
kpi = gold.groupby(['site', 'metier'], as_index=False).agg(
    effectif_reel=('effectif_reel', 'sum'),
    effectif_planifie=('effectif_planifie', 'sum'),
    departs_volontaires=('departs_volontaires', 'sum'),
    tension_recrutement=('tension_recrutement', 'mean'),
    couverture_competences_critiques=('couverture_competences_critiques', 'mean'),
    indicateur_risque_collectif=('indicateur_risque_collectif', 'mean')
)
kpi['ecart_effectif'] = kpi['effectif_reel'] - kpi['effectif_planifie']
display(kpi.sort_values('indicateur_risque_collectif', ascending=False).head(10))

## 7. Preparation modelisation

La cible est le volume agrege de departs volontaires. Le modele ne produit aucun score individuel.

In [ ]:
gold['mois'] = pd.to_datetime(gold['mois'])
gold['mois_num'] = gold['mois'].dt.year * 12 + gold['mois'].dt.month
features = ['site','metier','effectif_reel','effectif_planifie','ecart_effectif','recrutements_ouverts','tension_recrutement','couverture_competences_critiques','engagement_score_avg','training_hours_avg','absenteeism_rate','time_to_fill_days','offer_acceptance_rate','role_critique','mois_num']
target = 'departs_volontaires'
X = gold[features]
y = gold[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 8. Baseline et deux modeles predictifs

On compare une baseline moyenne, une regression lineaire et une Random Forest.

In [ ]:
cat_cols = ['site', 'metier']
num_cols = [c for c in features if c not in cat_cols]
prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), ('num', StandardScaler(), num_cols)])
models = {
    'Baseline moyenne': None,
    'Regression lineaire': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=160, max_depth=8, random_state=42)
}
rows = []
for name, model in models.items():
    if model is None:
        pred = np.repeat(y_train.mean(), len(y_test))
    else:
        pipe = Pipeline([('prep', prep), ('model', model)])
        pipe.fit(X_train, y_train)
        pred = np.maximum(pipe.predict(X_test), 0)
    rows.append({'modele': name, 'MAE': mean_absolute_error(y_test, pred), 'RMSE': mean_squared_error(y_test, pred) ** 0.5, 'R2': r2_score(y_test, pred)})
results = pd.DataFrame(rows).sort_values('MAE')
display(results)

## 9. Export des fichiers finaux

Les exports finaux sont disponibles dans `data/processed`.

In [ ]:
for path in sorted(PROCESSED.glob('*.csv')):
    print(path.name)